In [1]:
# --- Setup: base imports ---
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import inspect
import html
from IPython.display import display, HTML

# --- Setup: project path ---
# If the notebook is in L2_Regression/notebooks,
# this adds L2_Regression to PYTHONPATH
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# --- Imports from DM_org_refactored (Rough_programs) ---
from Rough_programs.DesignMatrix_refactored import (
    setup_market_parameters,
    setup_legendre_basis,
    generate_training_paths,
    rescale_data,
    build_design_matrix_legendre,
    build_response_vector,
)


# Optional (recommended) for syntax highlighting
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import HtmlFormatter
# Inject CSS once
_PYGMENTS_CSS_LOADED = False

def show_full_function(func, name=None):
    global _PYGMENTS_CSS_LOADED
    
    name = name or func.__name__
    source = inspect.getsource(func)

    # Syntax highlight HTML
    formatter = HtmlFormatter(style="friendly", noclasses=False)
    highlighted = highlight(source, PythonLexer(), formatter)

    # Load CSS only once
    if not _PYGMENTS_CSS_LOADED:
        css = formatter.get_style_defs('.highlight')
        display(HTML(f"<style>{css}</style>"))
        _PYGMENTS_CSS_LOADED = True

    html_block = f"""
    <details>
      <summary><strong>View full function: {html.escape(name)}</strong></summary>
      {highlighted}
    </details>
    """
    display(HTML(html_block))

<h1 style="text-align: center;">Single Level $L^2$ Regression using Markovian Projection</h1>

## Black-Scholes Correlated Model:
$$
dX(t)=a(t,X(t))\,dt + b(t,X(t))\,dW(t), \qquad 0<t<T,\quad X(t)\in\mathbb{R}^d,\quad W(t)\ \text{is a $k$-dim.\ Brownian motion}.
$$
$$
X(0)=x_0.
$$

- $a(t,x)=r\,x,\qquad r=0.05.$  
- $b(t,x)\in\mathbb{R}^{d\times k}.$  
- $(b_{\mathrm{BS}}(t,x))_{ij}=x_i\,\Sigma_{ij}.$  
- $P_1=(1,1,1)$

Also,
$$
\Sigma_{ij}=\sigma_i\,G_{ij}, \qquad 
GG^\top=
\begin{pmatrix}
1 & 0.8 & 0.3\\
0.8 & 1 & 0.1\\
0.3 & 0.1 & 1
\end{pmatrix},
\qquad
\sigma=(0.2,\ 0.15,\ 0.1)^\top.
$$
A common implementation uses an independent $k$-dimensional Brownian motion $W(t)$ and puts dependence in the diffusion matrix:
$$
dX(t)=rX(t)\,dt+\mathrm{diag}(X(t))\,\Sigma\,dW(t),\qquad \Sigma\in\mathbb{R}^{d\times k}.
$$
Componentwise,
$$
dX_i(t)=r X_idt+\sum_{j=1}^{k}X_i\Sigma_{ij}\,dW_j(t).
$$
Even if $dW_1,\ldots,dW_k$ are independent, assets $i$ and $\ell$ become correlated because they share the same factors $dW_j$ with weights $\Sigma_{ij}$ and $\Sigma_{\ell j}$. Indeed
$$
\mathbb{E}\!\left[\frac{dX_i(t)}{X_i(t)}\,\frac{dX_\ell(t)}{X_\ell(t)}\right]
=(\Sigma\Sigma^\top)_{i\ell}\,dt.
$$
In particular, if $\Sigma_{ij}=\sigma_i G_{ij}$, then $\sigma_i$ scales the volatility of asset $i$ and $G$ mixes the factors; the matrix $\Sigma\Sigma^\top$ determines the return covariance (and hence correlation) structure.

### Example ($d=3$)

Let $x=(x_1,x_2,x_3)^\top$ and $\sigma=(\sigma_1,\sigma_2,\sigma_3)^\top$. Assume the correlated Black–Scholes diffusion is defined by
$$
b_{\mathrm{BS}}(t,x)=\mathrm{diag}(x)\,\mathrm{diag}(\sigma)\,G,
$$
where $G\in\mathbb{R}^{3\times 3}$ is a factor-mixing matrix. First,
$$
\mathrm{diag}(x)\,\mathrm{diag}(\sigma)
=
\begin{pmatrix}
\sigma_1 x_1 & 0 & 0\\
0 & \sigma_2 x_2 & 0\\
0 & 0 & \sigma_3 x_3
\end{pmatrix}.
$$

Therefore,
$$
b_{\mathrm{BS}}(t,x)=
\begin{pmatrix}
\sigma_1 x_1 & 0 & 0\\
0 & \sigma_2 x_2 & 0\\
0 & 0 & \sigma_3 x_3
\end{pmatrix}
\begin{pmatrix}
G_{11} & G_{12} & G_{13}\\
G_{21} & G_{22} & G_{23}\\
G_{31} & G_{32} & G_{33}
\end{pmatrix}
=
\begin{pmatrix}
\sigma_1 x_1 G_{11} & \sigma_1 x_1 G_{12} & \sigma_1 x_1 G_{13}\\
\sigma_2 x_2 G_{21} & \sigma_2 x_2 G_{22} & \sigma_2 x_2 G_{23}\\
\sigma_3 x_3 G_{31} & \sigma_3 x_3 G_{32} & \sigma_3 x_3 G_{33}
\end{pmatrix}.
$$

Equivalently, each entry is
$$
\bigl(b_{\mathrm{BS}}(t,x)\bigr)_{ij}=\sigma_i x_i\,G_{ij}.
$$
With $d=3$ and $k=3$, the SDE
$$
dX(t)=rX(t)\,dt + b_{\mathrm{BS}}(t,X(t))\,dW(t),
\qquad W(t)=(W_1(t),W_2(t),W_3(t))^\top,
$$
reads componentwise as
$$
\left\{
\begin{aligned}
dX_1(t) &= rX_1(t)\,dt
+ \sigma_1 X_1(t)\Big(G_{11}\,dW_1(t)+G_{12}\,dW_2(t)+G_{13}\,dW_3(t)\Big),\\[2mm]
dX_2(t) &= rX_2(t)\,dt
+ \sigma_2 X_2(t)\Big(G_{21}\,dW_1(t)+G_{22}\,dW_2(t)+G_{23}\,dW_3(t)\Big),\\[2mm]
dX_3(t) &= rX_3(t)\,dt
+ \sigma_3 X_3(t)\Big(G_{31}\,dW_1(t)+G_{32}\,dW_2(t)+G_{33}\,dW_3(t)\Big).
\end{aligned}
\right.
$$

## Non-Markovian Process: 
Let $S(t)=P_1 X(t)$, with $P_1\in\mathbb{R}^{1\times d}$ and $X(t)\in\mathbb{R}^{d\times 1}$. Define the Markovian surrogate $\bar S^{(x_0)}(t)$ by
$$
\left\{
\begin{aligned}
d\bar S^{(x_0)}(t) &= \bar a^{(x_0)}\!\bigl(t,\bar S^{(x_0)}(t)\bigr)\,dt
+ \bar b^{(x_0)}\!\bigl(t,\bar S^{(x_0)}(t)\bigr)\,d\bar W(t), \qquad t\in[0,T],\\
\bar S^{(x_0)}(0) &= P_1 x_0.
\end{aligned}
\right.
$$

The drift and diffusion coefficients are given by conditional expectations:
$$
\bar a^{(x_0)}(t,s)
=\mathbb{E}\!\left[\,P_1 a\!\bigl(t,X(t)\bigr)\ \middle|\ P_1 X(t)=s,\ X(0)=x_0\,\right]
=\mathbb{E}\!\left[\,r\,P_1 X(t)\ \middle|\ P_1 X(t)=s,\ X(0)=x_0\,\right]
= r\,s,
$$
and
$$
\bigl(\bar b^{(x_0)}(t,s)\bigr)^2
=\mathbb{E}\!\left[\,P_1 b\!\bigl(t,X(t)\bigr)\,b\!\bigl(t,X(t)\bigr)^\top P_1^\top\ \middle|\ P_1 X(t)=s,\ X(0)=x_0\,\right].
$$
We keep the correlated Black–Scholes convention
$$
\bigl(b_{\mathrm{BS}}(t,x)\bigr)_{ij}=x_i\,\Sigma_{ij},
\qquad
\Sigma_{ij}=\sigma_i\,G_{ij},
$$
so that, in matrix form,
$$
b_{\mathrm{BS}}(t,x)=\mathrm{diag}(x)\,\Sigma
=\mathrm{diag}(x)\,\mathrm{diag}(\sigma)\,G.
$$

$$
b_{\mathrm{BS}}(t,x)\,b_{\mathrm{BS}}(t,x)^\top
=
\mathrm{diag}(x)\,\Sigma\Sigma^\top\,\mathrm{diag}(x).
$$
Visually
$$
D_x := \mathrm{diag}(x)=
\begin{pmatrix}
x_1 & 0 & 0\\
0 & x_2 & 0\\
0 & 0 & x_3
\end{pmatrix},
\qquad
C := \Sigma\Sigma^\top=
\begin{pmatrix}
c_{11} & c_{12} & c_{13}\\
c_{21} & c_{22} & c_{23}\\
c_{31} & c_{32} & c_{33}
\end{pmatrix}.
$$
Then
$$
b_{\mathrm{BS}}b_{\mathrm{BS}}^\top = D_x\,C\,D_x.
$$
$$
D_x\,C=
\begin{pmatrix}
x_1 c_{11} & x_1 c_{12} & x_1 c_{13}\\
x_2 c_{21} & x_2 c_{22} & x_2 c_{23}\\
x_3 c_{31} & x_3 c_{32} & x_3 c_{33}
\end{pmatrix}.
$$
Each row $i$ is multiplied by $x_i$.
$$
(D_x\,C)\,D_x=
\begin{pmatrix}
x_1^2 c_{11} & x_1 x_2 c_{12} & x_1 x_3 c_{13}\\
x_2 x_1 c_{21} & x_2^2 c_{22} & x_2 x_3 c_{23}\\
x_3 x_1 c_{31} & x_3 x_2 c_{32} & x_3^2 c_{33}
\end{pmatrix}.
$$
Each column $\ell$ is multiplied by $x_\ell$.
From the final matrix,
$$
(D_x\,C\,D_x)_{i\ell}=x_i\,x_\ell\,c_{i\ell}.
$$
Since $c_{i\ell}=(\Sigma\Sigma^\top)_{i\ell}$, we obtain
$$
\bigl(b_{\mathrm{BS}}b_{\mathrm{BS}}^\top\bigr)_{i\ell}
=
x_i x_\ell\,(\Sigma\Sigma^\top)_{i\ell}.
$$
Since $\Sigma=\mathrm{diag}(\sigma)G$, we have
$$
\Sigma\Sigma^\top
=
\mathrm{diag}(\sigma)\,GG^\top\,\mathrm{diag}(\sigma).
$$
Let $C:=GG^\top$. Then
$$
(\Sigma\Sigma^\top)_{i\ell}=\sigma_i\sigma_\ell\,C_{i\ell},
\qquad\Rightarrow\qquad
\bigl(b_{\mathrm{BS}}b_{\mathrm{BS}}^\top\bigr)_{i\ell}
=
\sigma_i\sigma_\ell\,x_i x_\ell\,C_{i\ell}.
$$

Take $P\in\mathbb{R}^{1\times 3}$ and define the scalar quantity
$$
P\,b_{\mathrm{BS}}(t,x)\,b_{\mathrm{BS}}(t,x)^\top P^\top.
$$
Expanding the quadratic form,
$$
P\,b_{\mathrm{BS}}b_{\mathrm{BS}}^\top P^\top
=
\sum_{i=1}^3\sum_{\ell=1}^3 p_i\,p_\ell\,
\sigma_i\sigma_\ell\,x_i x_\ell\,C_{i\ell},
\qquad P=(p_1,p_2,p_3).
$$
Replacing $x_i$ by $X_i(t)$ and conditioning on $S(t)=PX(t)=s$ yields
$$
\bigl(\bar b^{x_0}(t,s)\bigr)^2
=
\mathbb{E}\!\left[
P\,b_{\mathrm{BS}}\!\bigl(t,X(t)\bigr)\,b_{\mathrm{BS}}\!\bigl(t,X(t)\bigr)^\top P^\top
\ \middle|\ S(t)=s,\ X(0)=x_0
\right].
$$
For $P=\frac{1}{d}(1,\ldots,1)$ and $C=GG^\top$, this becomes
$$
\bigl(\bar b^{x_0}(t,s)\bigr)^2
=
\mathbb{E}\!\left[
\frac{1}{d^2}\sum_{i=1}^{d}\sum_{j=1}^{d}
\sigma_i\sigma_j\,X_i(t)\,X_j(t)\,C_{ij}
\ \middle|\ S(t)=s,\ X(0)=x_0
\right],
$$
which is the desired index-form expression.

## $L^2$ Regression Problem

We approximate $\big(\bar b^{x_0}\big)^2(\cdot,\cdot)$ by solving the time-integrated $L^2$ problem
$$
\big(\bar b^{x_0}\big)^2(\cdot,\cdot)
\approx
\arg\min_{h\in V}\ 
\int_{0}^{T}
\mathbb{E}\!\left[
\left(
P\,b\!\bigl(t,X(t)\bigr)\,b\!\bigl(t,X(t)\bigr)^\top P^\top
- h\!\bigl(t,PX(t)\bigr)
\right)^2
\ \middle|\ X(0)=x_0
\right]dt.
$$

For the correlated Black–Scholes structure, with $C:=GG^\top$, we have
$$
P\,b_{\mathrm{BS}}\!\bigl(t,X(t)\bigr)\,b_{\mathrm{BS}}\!\bigl(t,X(t)\bigr)^\top P^\top
=
\frac{1}{d^2}\sum_{i=1}^{d}\sum_{j=1}^{d}
\sigma_i\sigma_j\,X_i(t)\,X_j(t)\,C_{ij}.
$$

For example, one may choose a parametric class
$$
V=\{\,h:\ h(t,x)=c_0+c_1 t+c_2 x\,\}.
$$
Discretizing time with a grid $0=t_0<t_1<\cdots<t_N=T$, an empirical Monte Carlo objective is
$$
\big(\bar b^{x_0}\big)^2(\cdot,\cdot)
\approx
\arg\min_{h\in V}\ 
\frac{1}{M}\sum_{m=1}^{M}\ \frac{1}{N}\sum_{n=1}^{N}
\left(
\frac{1}{d^2}\sum_{i=1}^{d}\sum_{j=1}^{d}
\sigma_i\sigma_j\,X_i^{(m)}(t_n)\,X_j^{(m)}(t_n)\,C_{ij}
- h\!\bigl(t_n,PX^{(m)}(t_n)\bigr)
\right)^2.
$$

**Ansatz.** We approximate the squared projected diffusion by a finite expansion
$$
\tilde b^2(t,s)\approx \sum_{p=1}^{P} c_p\,\psi_p(t,s).
$$

### Legendre Polynomials
Legendre polynomials $\{P_n(x)\}_{n\ge 0}$ are orthogonal on $[-1,1]$:
$$
\int_{-1}^{1} P_m(x)\,P_n(x)\,dx
=
\frac{2}{2n+1}\,\delta_{mn}.
$$

The first few polynomials are:
- $P_0(x)=1$,
- $P_1(x)=x$,
- $P_2(x)=\frac{1}{2}\,(3x^2-1)$,
- $P_3(x)=\frac{1}{2}\,(5x^3-3x)$.

### Orthonormalization
Define the orthonormalized Legendre polynomials by
$$
\tilde P_n(x):=\sqrt{\frac{2n+1}{2}}\,P_n(x).
$$
Then
$$
\int_{-1}^{1}\tilde P_m(x)\,\tilde P_n(x)\,dx=\delta_{mn}.
$$

### Domain Scaling
Legendre polynomials are defined on $[-1,1]$, but our variables live on different intervals:
- Time: $t\in[0,T]$,
- Basket/state: $s\in[s_{\min},s_{\max}]$.

We map these intervals to $[-1,1]$ via affine transformations:
$$
\tau(t)=\frac{2t}{T}-1\in[-1,1],
\qquad
\xi(s)=\frac{2(s-s_{\min})}{s_{\max}-s_{\min}}-1\in[-1,1].
$$

### Determining $s_{\text{min}}$ and $s_{\text{max}}$
To determine $s_{\min}$ and $s_{\max}$, run a pilot simulation with $M_0=10{,}000$ paths and compute:
- $s_{\min}$: the $1$st percentile of the basket values,
- $s_{\max}$: the $99$th percentile of the basket values.

### Basis Functions
We define bivariate basis functions using orthonormal Legendre polynomials on $[-1,1]$:
$$
\psi_{(i_1,i_2)}(t,s)
=
\tilde P_{i_1}\!\bigl(\tau(t)\bigr)\,\tilde P_{i_2}\!\bigl(\xi(s)\bigr),
\qquad p=(i_1,i_2)\in\Lambda.
$$

We use a **total-degree** index set
$$
\Lambda=\{(i_1,i_2)\in\mathbb{N}_0^2:\ i_1+i_2\le d_{\max}\}.
$$

**Examples.**
- $d_{\max}=0$: $(i_1,i_2)=(0,0)$ $\Rightarrow$ $1$ basis function (constant).
- $d_{\max}=1$: $(0,0),(0,1),(1,0)$ $\Rightarrow$ $3$ basis functions.
- $d_{\max}=2$: $(0,0),(0,1),(0,2),(1,0),(2,0),(1,1)$ $\Rightarrow$ $6$ basis functions.

**General count.** The number of basis functions is
$$
P=\frac{(d_{\max}+1)(d_{\max}+2)}{2}.
$$

## Reformulated $L^2$ Regression Problem
Using the basis $\{\psi_p\}_{p\in\Lambda}$ defined above, we restrict the search space to the finite-dimensional class
$$
V:=\left\{\,h:\ h(t,s)=\sum_{p\in\Lambda} c_p\,\psi_p(t,s),\ \ c\in\mathbb{R}^{|\Lambda|}\right\}.
$$

The discrete $L^2$ regression problem for the projected diffusion becomes
$$
\big(\bar b^{x_0}\big)^2(\cdot,\cdot)
\approx
\arg\min_{h\in V}\ 
\frac{1}{M}\frac{1}{N}\sum_{m=1}^{M}\sum_{n=1}^{N}
\left(
\frac{1}{d^2}\sum_{i=1}^{d}\sum_{j=1}^{d}
\sigma_i\sigma_j\,X_i^{(m)}(t_n)\,X_j^{(m)}(t_n)\,C_{ij}
-
\sum_{p\in\Lambda} c_p\,\psi_p\!\bigl(t_n,\,P X^{(m)}(t_n)\bigr)
\right)^2.
$$

This transforms the problem into a **linear least-squares** problem for the coefficient vector $c=(c_p)_{p\in\Lambda}$.

## `legvander` in Python
Given input nodes $x=(x_1,\ldots,x_m)$ and a maximum degree $n$, `legvander` returns the Vandermonde-like matrix
$$
V\in\mathbb{R}^{m\times (n+1)},
\qquad
V=
\begin{pmatrix}
P_0(x_1) & P_1(x_1) & \cdots & P_n(x_1)\\
P_0(x_2) & P_1(x_2) & \cdots & P_n(x_2)\\
\vdots   & \vdots   & \ddots & \vdots\\
P_0(x_m) & P_1(x_m) & \cdots & P_n(x_m)
\end{pmatrix}.
$$
Here $P_k(\cdot)$ denotes the (Legendre) polynomial of degree $k$ evaluated at the given nodes.

### Data layout (paths, time grid, and basket values)

Let $\{t_n\}_{n=0}^{N_t-1}$ be the time grid and assume we simulate $M_t$ independent paths. We store the simulated state process in an array
$$
X\in\mathbb{R}^{M_t\times N_t\times d},
\qquad
X_{m,n,:}=X^{(m)}(t_n)\in\mathbb{R}^d,
$$
where $m=0,\ldots,M_t-1$ indexes paths and $n=0,\ldots,N_t-1$ indexes time steps.

### Projection (basket) evaluation

Given a projection row vector $P_1\in\mathbb{R}^{1\times d}$, define the scalar basket
$$
S(t)=P_1 X(t).
$$
For each path and time step, we compute
$$
S_{m,n}:=P_1 X^{(m)}(t_n),
\qquad
S\in\mathbb{R}^{M_t\times N_t}.
$$

Example (for $d=2$): if $P_1=(1,1)$, then for any $(m,n)$,
$$
S_{m,n}=X^{(m)}_1(t_n)+X^{(m)}_2(t_n).
$$

### Choosing $s_{\min}$ and $s_{\max}$ via percentiles

To scale $s$ into $[-1,1]$, we first estimate a robust range $[s_{\min},s_{\max}]$ from a pilot sample of basket values (e.g., $M_0$ paths):
- $s_{\min}$ = $1$st percentile of $\{S_{m,n}\}$,
- $s_{\max}$ = $99$th percentile of $\{S_{m,n}\}$.

(Interpretation: using percentiles avoids extreme outliers and yields a stable scaling interval.)

### Scaling the basket to $[-1,1]$

Once $s_{\min}$ and $s_{\max}$ are fixed, we map basket values to $[-1,1]$ by
$$
\xi(s)=\frac{2(s-s_{\min})}{s_{\max}-s_{\min}}-1.
$$
Applying this entrywise gives the scaled array
$$
S^{\mathrm{scaled}}_{m,n}:=\xi(S_{m,n}),
\qquad
S^{\mathrm{scaled}}\in\mathbb{R}^{M_t\times N_t}.
$$
### Time scaling and the flattened time vector `t_vals`

We use the affine time scaling
$$
\tau(t)=\frac{2t}{T}-1\in[-1,1],
\qquad
\tau_n:=\tau(t_n).
$$
Thus, the scaled time grid is the vector
$$
\tau_{\mathrm{grid}}=(\tau_0,\ldots,\tau_{N_t-1})^\top\in\mathbb{R}^{N_t}.
$$

When building a *single* regression/design matrix over all $(m,n)$ pairs, it is convenient to flatten the data into vectors of length $K=M_tN_t$.  
In particular, we define the flattened time vector
$$
t_{\mathrm{vals}}\in\mathbb{R}^{K}
$$
by repeating the scaled time grid for each path:
$$
t_{\mathrm{vals}}
=
(\underbrace{\tau_0,\ldots,\tau_{N_t-1}}_{\text{path }1},\ 
\underbrace{\tau_0,\ldots,\tau_{N_t-1}}_{\text{path }2},\ 
\ldots,\ 
\underbrace{\tau_0,\ldots,\tau_{N_t-1}}_{\text{path }M_t})^\top.
$$

Equivalently, using the indexing $k=(m,n)$ with $m=1,\ldots,M_t$ and $n=0,\ldots,N_t-1$,
$$
t_{\mathrm{vals}}[k]=\tau_n.
$$

## Design Matrix
Using a Legendre basis, the Design matrix $D\in \mathbb{R}^{(M_t N_t)\times P}$ is given by 
$$
D_{m,n,p}=\varphi_p(t_n,S_{m,n})=\tilde{P}_{i_1}(\tau(t_n))\times \tilde{P}_{i_2}(\xi(S_{m,n}))
$$
To construct the Design matrix we falt the pairs $(m,n)$ in one index $k=0,\ldots,K-1$ with $K=M_tN_t$, for instance $k=mN_t+n$. Hence $D\in \mathbb{R}^{K\times P}$, with $P=|\Lambda|$.
Define $K:=M_tN_t$ and let $d_{\max}$ be the maximum polynomial degree. We build the two Vandermonde-like matrices

$$
V_T \in \mathbb{R}^{K\times(d_{\max}+1)},
\qquad
V_T[k,i]
=
P_i\!\bigl(t_{\mathrm{vals}}[k]\bigr),
\quad i=0,\ldots,d_{\max},
$$

and

$$
V_S \in \mathbb{R}^{K\times(d_{\max}+1)},
\qquad
V_S[k,i]
=
P_i\!\bigl(s_{\mathrm{vals}}[k]\bigr),
\quad i=0,\ldots,d_{\max}.
$$

Visually (rows grouped by path), we may write
$$
V_T=
\left[
\begin{array}{cccc}
P_0(t_{\mathrm{vals}}[0]) & P_1(t_{\mathrm{vals}}[0]) & \cdots & P_{d_{\max}}(t_{\mathrm{vals}}[0])\\
P_0(t_{\mathrm{vals}}[1]) & P_1(t_{\mathrm{vals}}[1]) & \cdots & P_{d_{\max}}(t_{\mathrm{vals}}[1])\\
\vdots & \vdots & \ddots & \vdots\\
P_0(t_{\mathrm{vals}}[N_t-1]) & P_1(t_{\mathrm{vals}}[N_t-1]) & \cdots & P_{d_{\max}}(t_{\mathrm{vals}}[N_t-1])\\ \hline
P_0(t_{\mathrm{vals}}[N_t]) & P_1(t_{\mathrm{vals}}[N_t]) & \cdots & P_{d_{\max}}(t_{\mathrm{vals}}[N_t])\\
\vdots & \vdots & \ddots & \vdots\\
P_0(t_{\mathrm{vals}}[2N_t-1]) & P_1(t_{\mathrm{vals}}[2N_t-1]) & \cdots & P_{d_{\max}}(t_{\mathrm{vals}}[2N_t-1])\\ \hline
\vdots & \vdots & \ddots & \vdots\\ \hline
P_0(t_{\mathrm{vals}}[(M_t-1)N_t]) & P_1(t_{\mathrm{vals}}[(M_t-1)N_t]) & \cdots & P_{d_{\max}}(t_{\mathrm{vals}}[(M_t-1)N_t])\\
\vdots & \vdots & \ddots & \vdots\\
P_0(t_{\mathrm{vals}}[M_tN_t-1]) & P_1(t_{\mathrm{vals}}[M_tN_t-1]) & \cdots & P_{d_{\max}}(t_{\mathrm{vals}}[M_tN_t-1])
\end{array}
\right]
\ \ \left\}\ \begin{array}{l}
\text{rows }0\!:\!N_t-1 \ \text{(path 1)}\\
\text{rows }N_t\!:\!2N_t-1 \ \text{(path 2)}\\
\vdots\\
\text{rows }(M_t-1)N_t\!:\!M_tN_t-1 \ \text{(path $M_t$)}
\end{array}
\right.
$$

and similarly
$$
V_S=
\left[
\begin{array}{cccc}
P_0(s_{\mathrm{vals}}[0]) & P_1(s_{\mathrm{vals}}[0]) & \cdots & P_{d_{\max}}(s_{\mathrm{vals}}[0])\\
P_0(s_{\mathrm{vals}}[1]) & P_1(s_{\mathrm{vals}}[1]) & \cdots & P_{d_{\max}}(s_{\mathrm{vals}}[1])\\
\vdots & \vdots & \ddots & \vdots\\
P_0(s_{\mathrm{vals}}[N_t-1]) & P_1(s_{\mathrm{vals}}[N_t-1]) & \cdots & P_{d_{\max}}(s_{\mathrm{vals}}[N_t-1])\\ \hline
P_0(s_{\mathrm{vals}}[N_t]) & P_1(s_{\mathrm{vals}}[N_t]) & \cdots & P_{d_{\max}}(s_{\mathrm{vals}}[N_t])\\
\vdots & \vdots & \ddots & \vdots\\
P_0(s_{\mathrm{vals}}[2N_t-1]) & P_1(s_{\mathrm{vals}}[2N_t-1]) & \cdots & P_{d_{\max}}(s_{\mathrm{vals}}[2N_t-1])\\ \hline
\vdots & \vdots & \ddots & \vdots\\ \hline
P_0(s_{\mathrm{vals}}[(M_t-1)N_t]) & P_1(s_{\mathrm{vals}}[(M_t-1)N_t]) & \cdots & P_{d_{\max}}(s_{\mathrm{vals}}[(M_t-1)N_t])\\
\vdots & \vdots & \ddots & \vdots\\
P_0(s_{\mathrm{vals}}[M_tN_t-1]) & P_1(s_{\mathrm{vals}}[M_tN_t-1]) & \cdots & P_{d_{\max}}(s_{\mathrm{vals}}[M_tN_t-1])
\end{array}
\right]
\ \ \left\}\ \begin{array}{l}
\text{rows }0\!:\!N_t-1 \ \text{(path 1)}\\
\text{rows }N_t\!:\!2N_t-1 \ \text{(path 2)}\\
\vdots\\
\text{rows }(M_t-1)N_t\!:\!M_tN_t-1 \ \text{(path $M_t$)}
\end{array}
\right.
$$

For any $p=(i_1,i_2)$, the column $p$ of $D$ is:
$$
D[:,p]=VT[:,i_1]\odot VS[:,i_2]
$$
where $\odot$ denotes element-wise multiplication.

## Target Vector
For each path $m$ and time index $n$, define the target value
$$
\psi_{m,n}
=
\frac{1}{d^2}\sum_{i=1}^{d}\sum_{j=1}^{d}
\sigma_i\sigma_j\,X_i^{(m)}(t_n)\,X_j^{(m)}(t_n)\,C_{ij}.
$$
Equivalently, $\psi$ is constructed from the scalar quantity $P\,b\,b^\top P^\top$ evaluated along the simulated paths.

## Solving the linear system (least squares)

We solve the least-squares problem
$$
D\,c \approx \psi.
$$
Using a QR decomposition,
$$
D=Q R
\quad\Longrightarrow\quad
Q R c \approx \psi
\quad\Longrightarrow\quad
R c \approx Q^\top \psi,
$$
where $R$ is upper triangular.


<h1 style="text-align: center;">Implementation</h1>

<h4 style="text-align: left; color: #0066CC; font-weight: 700; margin-top: 14px;">
Implementation — Step 1: Market setup and Legendre basis definition
</h4>

In this first step, we initialize the advanced market model and define the polynomial basis used for the $L^2$ regression.

The underlying basket model is a correlated GBM system:
$$
dX_i(t)=rX_i(t)\,dt+\sigma_iX_i(t)\,(G\,dW(t))_i,\qquad i=1,\dots,d,
$$
where $G$ is the Cholesky factor of the correlation matrix $\Sigma$ ($\Sigma=GG^\top$).

We also define the projection weights
$$
w=\frac{1}{d}(1,\dots,1)^\top,\qquad S(t)=w^\top X(t).
$$

For regression, we use a tensor-product Legendre basis:
$$
\varphi_{ij}(t,s)=\widetilde P_i(t)\widetilde P_j(s),
\qquad
(i,j)\in\{0,\dots,\text{max\_deg}_t\}\times\{0,\dots,\text{max\_deg}_s\}.
$$

<details>
<summary><strong>View function used (syntax + explanation): setup_market_parameters</strong></summary>

**Function:** `setup_market_parameters()`

**Location:** `L2_Regression/Rough_programs/DesignMatrix_refactored.py`

**What it does:**
- Defines market and simulation parameters for the advanced setting:
  - correlated assets,
  - heterogeneous volatilities,
  - equal-weight basket.
- Returns a dictionary `params` containing:
  - `d`, `initial_prices`, `r`, `vol`,
  - `cov_mat`, `G`,
  - `dt`, `sqrt_dt`,
  - `basket_weights`, `num_time_steps`, `num_training_paths`.

<pre><code>params = setup_market_parameters()</code></pre>

</details>

<details>
<summary><strong>View function used (syntax + explanation): setup_legendre_basis</strong></summary>

**Function:** `setup_legendre_basis(max_deg_t, max_deg_s)`

**Location:** `L2_Regression/Rough_programs/DesignMatrix_refactored.py`

**What it does:**
- Builds the index set for tensor-product Legendre basis functions.
- Returns:
  - `basis_pairs`,
  - `num_basis_functions`,
  - `max_deg_t`,
  - `max_deg_s`.

<pre><code>basis_pairs, num_basis_functions, max_deg_t, max_deg_s = setup_legendre_basis(max_deg_t, max_deg_s)</code></pre>

</details>


In [23]:
show_full_function(setup_market_parameters)

In [24]:
show_full_function(setup_legendre_basis)

In [25]:
# Step 1 implementation
params = setup_market_parameters()

max_deg_t = 3
max_deg_s = 3
basis_pairs, num_basis_functions, max_deg_t, max_deg_s = setup_legendre_basis(max_deg_t, max_deg_s)

print("\n--- Step 1 summary ---")
print("d =", params["d"])
print("r =", params["r"])
print("vol =", params["vol"])
print("cov_mat =\n", params["cov_mat"])
print("num_time_steps =", params["num_time_steps"])
print("num_training_paths =", params["num_training_paths"])
print("num_basis_functions =", num_basis_functions)
print("first basis pairs =", basis_pairs[:8])


Market Setup:
  Number of assets: 3
  Volatilities: [0.2  0.15 0.1 ]
  Correlation matrix:
[[1.  0.8 0.3]
 [0.8 1.  0.1]
 [0.3 0.1 1. ]]
  Training paths: 200

Legendre Basis Setup:
  Max degree (time): 3
  Max degree (space): 3
  Number of basis functions: 16
  Basis pairs: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 0), (1, 1)]... (showing first 6)

--- Step 1 summary ---
d = 3
r = 0.05
vol = [0.2  0.15 0.1 ]
cov_mat =
 [[1.  0.8 0.3]
 [0.8 1.  0.1]
 [0.3 0.1 1. ]]
num_time_steps = 100
num_training_paths = 200
num_basis_functions = 16
first basis pairs = [(0, 0), (0, 1), (0, 2), (0, 3), (1, 0), (1, 1), (1, 2), (1, 3)]


<h4 style="text-align: left; color: #0066CC; font-weight: 700; margin-top: 14px;">
Implementation — Step 2: Generate correlated training paths
</h4>

In this step, we simulate $M$ training trajectories of the correlated $d$-dimensional model:
$$
dX_i(t)=rX_i(t)\,dt+\sigma_iX_i(t)\,(G\,dW(t))_i,\qquad i=1,\dots,d,
$$
where $G$ is the Cholesky factor of the correlation matrix $\Sigma$.

Using Euler–Maruyama on $t_n=n\Delta t$:
$$
X^{n+1}=X^n+rX^n\Delta t+\operatorname{diag}(\sigma\odot X^n)\,(GZ^n)\sqrt{\Delta t},
\qquad Z^n\sim\mathcal N(0,I_d).
$$

This produces:
- `stock_price_paths` with shape $(M, N_t, d)$,
- `time_grid` with shape $(N_t,)$.

These trajectories are the raw dataset used in the following steps to build the regression inputs.

<details>
<summary><strong>View function used (syntax + explanation): generate_training_paths</strong></summary>

**Function:** `generate_training_paths(params)`

**Location:** `L2_Regression/Rough_programs/DesignMatrix_refactored.py`

**What it does:**
- Simulates correlated GBM paths using Cholesky-induced correlation.
- Returns all time states (not only terminal values).

<pre><code>stock_price_paths, time_grid = generate_training_paths(params)</code></pre>

</details>


In [26]:
show_full_function(generate_training_paths)

In [16]:
# Step 2 implementation
stock_price_paths, time_grid = generate_training_paths(params)

print("\n--- Step 2 summary ---")
print("stock_price_paths shape:", stock_price_paths.shape)  # expected: (M, N_t, d)
print("time_grid shape:", time_grid.shape)                  # expected: (N_t,)
print("first 5 time points:", time_grid[:5])
print("first path, first time state:", stock_price_paths[0, 0, :])
print("first path, last time state :", stock_price_paths[0, -1, :])


Generating 200 correlated training paths...
Training data generation complete!

--- Step 2 summary ---
stock_price_paths shape: (200, 100, 3)
time_grid shape: (100,)
first 5 time points: [0.         0.01010101 0.02020202 0.03030303 0.04040404]
first path, first time state: [231.34287086 255.82043412 272.23931986]
first path, last time state : [291.40747513 325.42645245 263.08889162]


<h4 style="text-align: left; color: #0066CC; font-weight: 700; margin-top: 14px;">
Implementation  Step 3: Rescale $(t,s)$ data for Legendre regression
</h4>

Before building the Legendre design matrix, we construct the projected basket values
$$
S_n^m = w^\top X^m(t_n),
\qquad w=\frac1d(1,\dots,1)^\top,
$$
and flatten the dataset over all pairs $(m,n)$.

Since Legendre polynomials are naturally stable on bounded normalized domains, we rescale
time and basket coordinates using
$$
t_{\mathrm{sc}}=\frac{t-\mu_t}{\sigma_t},
\qquad
s_{\mathrm{sc}}=\frac{s-\mu_s}{\sigma_s}.
$$

This produces normalized vectors $(t_{\mathrm{sc}}, s_{\mathrm{sc}})$ used in the next step to evaluate tensor-product Legendre basis functions.

<details>
<summary><strong>View function used (syntax + explanation): rescale_data</strong></summary>

**Function:** `rescale_data(stock_price_paths, time_grid, params)`

**Location:** `L2_Regression/Rough_programs/DesignMatrix_refactored.py`

**What it does:**
- Builds flattened time and basket samples across all paths and time steps.
- Computes mean/std for each coordinate.
- Returns scaled vectors and scaling parameters.

<pre><code>t_vals, s_vals, t_mean, t_std, s_mean, s_std = rescale_data(stock_price_paths, time_grid, params)</code></pre>

</details>


In [28]:
show_full_function(rescale_data)

In [29]:
# Step 3 implementation
t_vals, s_vals, t_mean, t_std, s_mean, s_std = rescale_data(
    stock_price_paths, time_grid, params
)

print("\n--- Step 3 summary ---")
print("t_vals shape:", t_vals.shape)
print("s_vals shape:", s_vals.shape)
print("t_mean, t_std:", t_mean, t_std)
print("s_mean, s_std:", s_mean, s_std)
print("scaled t range:", (t_vals.min(), t_vals.max()))
print("scaled s range:", (s_vals.min(), s_vals.max()))


Data Rescaling:
  Time range (original): [0.000, 1.000]
  Time range (scaled): [-1.715, 1.715]
  Basket range (original): [187.3, 328.1]
  Basket range (scaled): [-3.221, 4.978]

--- Step 3 summary ---
t_vals shape: (20000,)
s_vals shape: (20000,)
t_mean, t_std: 0.5 0.29157646512850627
s_mean, s_std: 257.70184396867955 23.47413695820653
scaled t range: (np.float64(-1.7148160424389376), np.float64(1.7148160424389376))
scaled s range: (np.float64(-3.2209421490865173), np.float64(4.977796152486369))


<h4 style="text-align: left; color: #0066CC; font-weight: 700; margin-top: 14px;">
Implementation — Step 4: Build the Legendre design matrix $D$
</h4>

Using the scaled variables $(t_{\mathrm{sc}}, s_{\mathrm{sc}})$ from Step 3, we now build the regression matrix $D$ with tensor-product orthonormal Legendre basis functions:
$$
\phi_{ij}(t,s)=\widetilde P_i(t)\widetilde P_j(s),
\qquad
\widetilde P_n(x)=\sqrt{\frac{2n+1}{2}}\,P_n(x).
$$

For each sample index $k \leftrightarrow (m,n)$ and basis index $p \leftrightarrow (i,j)$:
$$
D_{k,p}=\widetilde P_i\!\big(t_{\mathrm{sc},k}\big)\,\widetilde P_j\!\big(s_{\mathrm{sc},k}\big).
$$

Hence, the matrix has size
$$
D\in\mathbb{R}^{(M N_t)\times P},
\qquad
P=(\text{max\_deg}_t+1)(\text{max\_deg}_s+1).
$$

This step also prints empirical diagnostics related to orthogonality (via $D^\top D$), which helps assess numerical stability before solving the regression.

<details>
<summary><strong>View function used (syntax + explanation): build_design_matrix_legendre</strong></summary>

**Function:** `build_design_matrix_legendre(t_vals, s_vals, max_deg_t, max_deg_s, basis_pairs)`

**Location:** `L2_Regression/Rough_programs/DesignMatrix_refactored.py`

**What it does:**
- Evaluates normalized Legendre basis functions on scaled data.
- Builds the tensor-product design matrix.
- Reports empirical orthogonality diagnostics.

<pre><code>design_matrix = build_design_matrix_legendre(t_vals, s_vals, max_deg_t, max_deg_s, basis_pairs)</code></pre>

</details>


In [31]:
show_full_function(build_design_matrix_legendre)

In [32]:
# Step 4 implementation
design_matrix = build_design_matrix_legendre(
    t_vals=t_vals,
    s_vals=s_vals,
    max_deg_t=max_deg_t,
    max_deg_s=max_deg_s,
    basis_pairs=basis_pairs
)

print("\n--- Step 4 summary ---")
print("design_matrix shape:", design_matrix.shape)
print("expected rows M*N:", params["num_training_paths"] * params["num_time_steps"])
print("expected cols P:", len(basis_pairs))
print("any NaN in design_matrix?", np.isnan(design_matrix).any())
print("cond(D):", np.linalg.cond(design_matrix))



Building Legendre Design Matrix...
  Design matrix shape: (20000, 16)
  Checking orthogonality: DᵀD should be approximately identity...
  Expected diagonal value: 20000
  Actual diagonal value: 92627368
  Off-diagonal norm: 7.69e+08 (should be small)

--- Step 4 summary ---
design_matrix shape: (20000, 16)
expected rows M*N: 20000
expected cols P: 16
any NaN in design_matrix? False
cond(D): 754.1380227038219


<h4 style="text-align: left; color: #0066CC; font-weight: 700; margin-top: 14px;">
Implementation — Step 5: Build the response vector $\psi$
</h4>

In this step, we construct the regression target $\psi$, i.e., the true instantaneous basket variance at each data point $(m,n)$.

Let
$$
S_t = w^\top X_t,\qquad w=\frac1d(1,\dots,1)^\top.
$$
For correlated assets with volatility vector $\sigma=(\sigma_1,\dots,\sigma_d)$ and correlation matrix $\Sigma$, define
$$
A_t=\operatorname{diag}(\sigma_1X_1(t),\dots,\sigma_dX_d(t)).
$$
Then the basket variance is
$$
\psi(t,X_t)=w^\top A_t\,\Sigma\,A_t\,w.
$$

In discrete form, each flattened index $k\leftrightarrow(m,n)$ gets
$$
\psi_k = w^\top A_n^m\,\Sigma\,A_n^m\,w.
$$

This is the right-hand side of the $L^2$ regression problem:
$$
\min_c \|Dc-\psi\|_2^2.
$$

<details>
<summary><strong>View function used (syntax + explanation): build_response_vector</strong></summary>

**Function:** `build_response_vector(stock_price_paths, params)`

**Location:** `L2_Regression/Rough_programs/DesignMatrix_refactored.py`

**What it does:**
- Computes the true instantaneous basket variance at every $(m,n)$ point.
- Returns the flattened response vector $\psi\in\mathbb{R}^{M N_t}$.

<pre><code>response_vector = build_response_vector(stock_price_paths, params)</code></pre>

</details>


In [34]:
show_full_function(build_response_vector)

In [35]:
# Step 5 implementation
response_vector = build_response_vector(stock_price_paths, params)

print("\n--- Step 5 summary ---")
print("response_vector shape:", response_vector.shape)
print("expected length M*N:", params["num_training_paths"] * params["num_time_steps"])
print("any NaN in response_vector?", np.isnan(response_vector).any())
print("response min/max:", response_vector.min(), response_vector.max())
print("response mean/std:", response_vector.mean(), response_vector.std(ddof=1))


Building Response Vector...
  Response vector shape: (20000,)
  Mean variance: 941.621247
  Variance range: [386.588295, 2487.843490]

--- Step 5 summary ---
response_vector shape: (20000,)
expected length M*N: 20000
any NaN in response_vector? False
response min/max: 386.58829548036204 2487.843489868573
response mean/std: 941.6212473791238 222.7294539387904
